In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.io import mmread
import anndata as ad
import gc

base_dir = Path(r"C:\Users\aabassetc2\Documents\github\benchmark-integration")

REDUCTION_MAP = {
    "unintegrated": {"embedding": "pca",              "umap": "umap",         "obsm_key": "X_pca"},
    "cca":          {"embedding": "cca",               "umap": "umap_cca",    "obsm_key": "X_cca"},
    "rpca":         {"embedding": "rpca",              "umap": "umap_rpca",   "obsm_key": "X_rpca"},
    "harmony":      {"embedding": "harmony",           "umap": "umap_harmony","obsm_key": "X_harmony"},
    "scvi":         {"embedding": "integrated_scvi",   "umap": "umap",        "obsm_key": "X_scvi"},
}

methods_to_process = ["unintegrated", "cca", "rpca", "harmony", "scvi"]

print("Setup complete!")
print(f"Base dir: {base_dir}")
print(f"Methods: {methods_to_process}")

In [ ]:
for integration_method in methods_to_process:

    print(f"\n{'='*60}")
    print(f"Processing: {integration_method}")
    print(f"{'='*60}")

    input_dir = base_dir / "data" / "output" / integration_method
    reduction_config = REDUCTION_MAP[integration_method]

    # ------------------------------------------------------------------
    # Read metadata
    # ------------------------------------------------------------------

    cell_metadata = pd.read_csv(input_dir / "cell_metadata.csv")
    cell_names = pd.read_csv(input_dir / "cells.csv")
    cell_metadata.index = cell_names["cell"].values

    gene_names = pd.read_csv(input_dir / "genes.csv")
    try:
        gene_mapping_table = pd.read_csv(input_dir / "gene_mapping.csv")
        gene_mapping_table.index = gene_names["gene"].values
    except FileNotFoundError:
        print("WARNING: gene_mapping.csv not found, using gene names only")
        gene_mapping_table = gene_names.set_index("gene", drop=False)

    # ------------------------------------------------------------------
    # Read embeddings
    # ------------------------------------------------------------------

    reductions_dir = input_dir / "reductions"
    embedding_file = reductions_dir / f"{reduction_config['embedding']}.csv"
    umap_file = reductions_dir / f"{reduction_config['umap']}.csv"

    print(f"Embedding: {embedding_file.name} (exists: {embedding_file.exists()})")
    print(f"UMAP: {umap_file.name} (exists: {umap_file.exists()})")

    embedding = pd.read_csv(embedding_file) \
        .set_index(pd.Index(cell_names['cell'])) \
        .to_numpy()

    umap_embedding = pd.read_csv(umap_file) \
        .set_index(pd.Index(cell_names['cell'])) \
        .to_numpy()

    print(f"Embedding shape: {embedding.shape}")
    print(f"UMAP shape: {umap_embedding.shape}")

    # PCA reference (for integrated methods)
    pca_embedding = None
    if reduction_config["embedding"] != "pca":
        pca_file = reductions_dir / "pca.csv"
        if pca_file.exists():
            pca_embedding = pd.read_csv(pca_file) \
                .set_index(pd.Index(cell_names['cell'])) \
                .to_numpy()
            print(f"PCA (reference) shape: {pca_embedding.shape}")
    else:
        pca_embedding = embedding

    # ------------------------------------------------------------------
    # Read KNN (optional)
    # ------------------------------------------------------------------

    knn_idx = knn_dist_mat = knn_conn_mat = None
    if (input_dir / "KNN_indices.csv").exists():
        knn_idx = pd.read_csv(input_dir / "KNN_indices.csv").to_numpy().astype('int32') - 1
    if (input_dir / "KNN_distances.mtx").exists():
        knn_dist_mat = mmread(str(input_dir / "KNN_distances.mtx")).tocsr().astype('float64')
    if (input_dir / "UMAP_connectivity_matrix.mtx").exists():
        knn_conn_mat = mmread(str(input_dir / "UMAP_connectivity_matrix.mtx")).tocsr().astype('float64')

    # ------------------------------------------------------------------
    # Read SCT counts
    # ------------------------------------------------------------------

    counts_dir = input_dir / "sct-counts"
    SCT_counts = SCT_data = None

    if (counts_dir / "SCT_counts.mtx").exists():
        print("Loading SCT counts...")
        SCT_counts = mmread(str(counts_dir / "SCT_counts.mtx")).tocsr().astype('float64')
        if SCT_counts.shape[0] != len(cell_names):
            SCT_counts = SCT_counts.T
        print(f"SCT counts: {SCT_counts.shape}")

    if (counts_dir / "SCT_data.mtx").exists():
        print("Loading SCT data (normalized)...")
        SCT_data = mmread(str(counts_dir / "SCT_data.mtx")).tocsr().astype('float64')
        if SCT_data.shape[0] != len(cell_names):
            SCT_data = SCT_data.T
        print(f"SCT data: {SCT_data.shape}")

    # ------------------------------------------------------------------
    # Construct AnnData
    # ------------------------------------------------------------------

    if SCT_data is not None:
        X = SCT_data
        print("Using SCT normalized data as X")
    elif SCT_counts is not None:
        X = SCT_counts
        print("Using SCT counts as X")
    else:
        print("WARNING: No count matrix found!")
        X = None

    obsm_key = reduction_config["obsm_key"]
    dimred_dict = {obsm_key: embedding, "X_umap": umap_embedding}
    if pca_embedding is not None and reduction_config["embedding"] != "pca":
        dimred_dict["X_pca"] = pca_embedding

    graph_dict = {}
    if knn_dist_mat is not None:
        graph_dict['distances'] = knn_dist_mat
    if knn_conn_mat is not None:
        graph_dict['connectivities'] = knn_conn_mat

    uns_dict = {'integration_method': integration_method}
    if knn_idx is not None:
        uns_dict['neighbors'] = {
            'connectivities_key': 'connectivities',
            'distances_key': 'distances',
            'indices': knn_idx,
            'params': {
                'n_neighbors': knn_idx.shape[1],
                'method': f'Seurat ({integration_method})',
                'metric': 'cosine',
                'n_pcs': embedding.shape[1],
                'use_rep': obsm_key,
            }
        }

    layers_dict = {}
    if SCT_counts is not None:
        layers_dict['counts'] = SCT_counts
    if SCT_data is not None and SCT_counts is not None:
        layers_dict['normalized'] = SCT_data

    adata = ad.AnnData(
        X=X,
        obs=cell_metadata,
        var=gene_mapping_table,
        layers=layers_dict if layers_dict else None,
        obsm=dimred_dict,
        obsp=graph_dict if graph_dict else None,
        uns=uns_dict,
    )

    print(f"\n{adata}")
    print(f"obsm keys: {list(adata.obsm.keys())}")

    # ------------------------------------------------------------------
    # Compute neighbors
    # ------------------------------------------------------------------

    print(f"Computing neighbors using {obsm_key}...")
    sc.pp.neighbors(adata, use_rep=obsm_key, n_neighbors=30, metric="cosine")

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 6))
    sc.pl.scatter(
        adata,
        basis='umap',
        title=f'{integration_method}',
        frameon=True,
        show=False,
        legend_fontsize=10,
        ax=ax,
    )
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------

    output_file = base_dir / "data" / "output" / f"{integration_method}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved to {output_file}")

    # Free memory
    del adata, X, SCT_counts, SCT_data, embedding, umap_embedding
    gc.collect()

In [ ]:
print(f"\n{'='*60}")
print("All methods processed!")
print(f"{'='*60}")

# List the output files
output_dir = base_dir / "data" / "output"
h5ad_files = list(output_dir.glob("*.h5ad"))
print(f"\nSaved {len(h5ad_files)} h5ad files:")
for f in sorted(h5ad_files):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name}  ({size_mb:.1f} MB)")